# Tres en Raya - Todos los métodos

Cada método sigue exactamente la misma estructura de `02_bandits.ipynb`.

In [ ]:
import numpy as np
import math
import matplotlib.pyplot as plt

In [ ]:
def movimientos_validos(state):
    return [(i, j) for j in range(3) for i in range(3) if state[i, j] == 0]

def juego_terminado(state):
    if (state.sum(axis=0) == 3).sum() >= 1 or (state.sum(axis=1) == 3).sum() >= 1:
        return 1
    if (state.sum(axis=0) == -3).sum() >= 1 or (state.sum(axis=1) == -3).sum() >= 1:
        return -1
    diag = [sum([state[i, i] for i in range(3)]), sum([state[i, 2 - i] for i in range(3)])]
    if diag[0] == 3 or diag[1] == 3:
        return 1
    if diag[0] == -3 or diag[1] == -3:
        return -1
    if len(movimientos_validos(state)) == 0:
        return 0
    return None

def estado_str(state):
    return str(state.reshape(3*3))

## Método ε-greedy

In [ ]:
partidas = 50
turnos = 100
epsilons = [0, 0.1, 0.5]
recompensas_medias = np.zeros((len(epsilons), turnos))

for ej in range(partidas):
    for i, e in enumerate(epsilons):
        V = {}
        for exp in range(turnos):
            state = np.zeros((3,3))
            positions = []
            turno = 1
            while True:
                valid = movimientos_validos(state)
                if not valid:
                    break
                if turno == 1:
                    if np.random.uniform(0, 1) < e:
                        ix = np.random.choice(len(valid))
                        row, col = valid[ix]
                    else:
                        max_val = -1000
                        best = valid[0]
                        for r, c in valid:
                            ns = state.copy()
                            ns[r, c] = 1
                            key = estado_str(ns)
                            val = V.get(key, 0)
                            if val >= max_val:
                                max_val = val
                                best = (r, c)
                        row, col = best
                    state[row, col] = 1
                    positions.append(estado_str(state))
                else:
                    ix = np.random.choice(len(valid))
                    row, col = valid[ix]
                    state[row, col] = -1
                result = juego_terminado(state)
                if result is not None:
                    break
                turno = -turno
            if result == 1:
                recompensa = 1
            elif result == 0:
                recompensa = 0.5
            else:
                recompensa = 0
            for p in reversed(positions):
                if V.get(p) is None:
                    V[p] = 0
                V[p] += 0.5 * (recompensa - V[p])
                recompensa = V[p]
            recompensas_medias[i][exp] += (result == 1)

recompensas_medias /= partidas

In [ ]:
plt.figure(figsize=(8, 5))
for i, e in enumerate(epsilons):
    plt.plot(recompensas_medias[i], label=f'$\\epsilon$ = {e}')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xlabel('Partidas')
plt.ylabel('Tasa de victorias')
plt.title('ε-greedy - Tres en raya')
plt.show()

## Método de valores iniciales optimistas

In [ ]:
partidas = 50
turnos = 100
valores_iniciales = [0.5, 1.0, 2.0]
recompensas_medias = np.zeros((len(valores_iniciales), turnos))

for ej in range(partidas):
    for i, v0 in enumerate(valores_iniciales):
        V = {}
        for exp in range(turnos):
            state = np.zeros((3,3))
            positions = []
            turno = 1
            while True:
                valid = movimientos_validos(state)
                if not valid:
                    break
                if turno == 1:
                    max_val = -1000
                    best = valid[0]
                    for r, c in valid:
                        ns = state.copy()
                        ns[r, c] = 1
                        key = estado_str(ns)
                        val = V.get(key, v0)
                        if val >= max_val:
                            max_val = val
                            best = (r, c)
                    row, col = best
                    state[row, col] = 1
                    positions.append(estado_str(state))
                else:
                    ix = np.random.choice(len(valid))
                    row, col = valid[ix]
                    state[row, col] = -1
                result = juego_terminado(state)
                if result is not None:
                    break
                turno = -turno
            if result == 1:
                recompensa = 1
            elif result == 0:
                recompensa = 0.5
            else:
                recompensa = 0
            for p in reversed(positions):
                if V.get(p) is None:
                    V[p] = v0
                V[p] += 0.5 * (recompensa - V[p])
                recompensa = V[p]
            recompensas_medias[i][exp] += (result == 1)

recompensas_medias /= partidas

In [ ]:
plt.figure(figsize=(8, 5))
for i, v0 in enumerate(valores_iniciales):
    plt.plot(recompensas_medias[i], label=f'V₀ = {v0}')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xlabel('Partidas')
plt.ylabel('Tasa de victorias')
plt.title('Valores iniciales optimistas - Tres en raya')
plt.show()

## Método UCB

In [ ]:
partidas = 50
turnos = 100
valores_c = [0.5, 1.0, 2.0]
recompensas_medias = np.zeros((len(valores_c), turnos))

for ej in range(partidas):
    for i, c in enumerate(valores_c):
        V = {}
        visits = {}
        t = 0
        for exp in range(turnos):
            state = np.zeros((3,3))
            positions = []
            turno = 1
            while True:
                valid = movimientos_validos(state)
                if not valid:
                    break
                t += 1
                if turno == 1:
                    max_val = -1000
                    best = valid[0]
                    for r, c in valid:
                        ns = state.copy()
                        ns[r, c] = 1
                        key = estado_str(ns)
                        q = V.get(key, 0)
                        n = visits.get(key, 1)
                        score = q + c * math.sqrt(math.log(t + 1) / n)
                        if score >= max_val:
                            max_val = score
                            best = (r, c)
                    row, col = best
                    state[row, col] = 1
                    sk = estado_str(state)
                    positions.append(sk)
                    visits[sk] = visits.get(sk, 0) + 1
                else:
                    ix = np.random.choice(len(valid))
                    row, col = valid[ix]
                    state[row, col] = -1
                result = juego_terminado(state)
                if result is not None:
                    break
                turno = -turno
            if result == 1:
                recompensa = 1
            elif result == 0:
                recompensa = 0.5
            else:
                recompensa = 0
            for p in reversed(positions):
                if V.get(p) is None:
                    V[p] = 0
                V[p] += 0.5 * (recompensa - V[p])
                recompensa = V[p]
            recompensas_medias[i][exp] += (result == 1)

recompensas_medias /= partidas

In [ ]:
plt.figure(figsize=(8, 5))
for i, c in enumerate(valores_c):
    plt.plot(recompensas_medias[i], label=f'c = {c}')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xlabel('Partidas')
plt.ylabel('Tasa de victorias')
plt.title('UCB - Tres en raya')
plt.show()

## Algoritmo de Gradiente

In [ ]:
def softmax(x):
    return np.exp(x)/sum(np.exp(x))

In [ ]:
partidas = 50
turnos = 100
alphas = [0.05, 0.1, 0.5]
recompensas_medias = np.zeros((len(alphas), turnos))

for ej in range(partidas):
    for i, lr in enumerate(alphas):
        H = {}
        rewards_history = []
        for exp in range(turnos):
            state = np.zeros((3,3))
            positions = []
            chosen_probs = []
            turno = 1
            while True:
                valid = movimientos_validos(state)
                if not valid:
                    break
                if turno == 1:
                    if len(valid) == 1:
                        row, col = valid[0]
                    else:
                        next_states = []
                        for r, c in valid:
                            ns = state.copy()
                            ns[r, c] = 1
                            next_states.append(estado_str(ns))
                        prefs = [H.get(s, 0) for s in next_states]
                        pi = softmax(prefs)
                        ix = np.random.choice(len(valid), p=pi)
                        row, col = valid[ix]
                        chosen_probs.append((estado_str(state), pi[ix]))
                    state[row, col] = 1
                    positions.append(estado_str(state))
                else:
                    ix = np.random.choice(len(valid))
                    row, col = valid[ix]
                    state[row, col] = -1
                result = juego_terminado(state)
                if result is not None:
                    break
                turno = -turno
            if result == 1:
                recompensa = 1
            elif result == 0:
                recompensa = 0.5
            else:
                recompensa = 0
            rewards_history.append(recompensa)
            r_mean = np.mean(rewards_history)
            for p in reversed(positions):
                if H.get(p) is None:
                    H[p] = 0
                pi_chosen = 0
                for s, pc in chosen_probs:
                    if s == p:
                        pi_chosen = pc
                        break
                H[p] += lr * (recompensa - r_mean) * (1 - pi_chosen)
                recompensa = H[p]
            recompensas_medias[i][exp] += (result == 1)

recompensas_medias /= partidas

In [ ]:
plt.figure(figsize=(8, 5))
for i, a in enumerate(alphas):
    plt.plot(recompensas_medias[i], label=f'$\\alpha$ = {a}')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xlabel('Partidas')
plt.ylabel('Tasa de victorias')
plt.title('Gradiente (Softmax) - Tres en raya')
plt.show()

## Comparación de métodos

In [ ]:
partidas = 50
turnos = 100
metodos = ['ε-greedy (ε=0.1)', 'Optimista (V₀=1.0)', 'UCB (c=1.0)', 'Gradiente (α=0.1)']
recompensas_medias = np.zeros((len(metodos), turnos))

for ej in range(partidas):
    # ε-greedy
    i = 0
    V = {}
    for exp in range(turnos):
        state = np.zeros((3,3))
        positions = []
        turno = 1
        while True:
            valid = movimientos_validos(state)
            if not valid:
                break
            if turno == 1:
                if np.random.uniform(0, 1) < 0.1:
                    ix = np.random.choice(len(valid))
                    row, col = valid[ix]
                else:
                    max_val = -1000
                    best = valid[0]
                    for r, c in valid:
                        ns = state.copy()
                        ns[r, c] = 1
                        key = estado_str(ns)
                        val = V.get(key, 0)
                        if val >= max_val:
                            max_val = val
                            best = (r, c)
                    row, col = best
                state[row, col] = 1
                positions.append(estado_str(state))
            else:
                ix = np.random.choice(len(valid))
                row, col = valid[ix]
                state[row, col] = -1
            result = juego_terminado(state)
            if result is not None:
                break
            turno = -turno
        if result == 1:
            recompensa = 1
        elif result == 0:
            recompensa = 0.5
        else:
            recompensa = 0
        for p in reversed(positions):
            if V.get(p) is None:
                V[p] = 0
            V[p] += 0.5 * (recompensa - V[p])
            recompensa = V[p]
        recompensas_medias[i][exp] += (result == 1)

for ej in range(partidas):
    # Optimista
    i = 1
    V = {}
    for exp in range(turnos):
        state = np.zeros((3,3))
        positions = []
        turno = 1
        while True:
            valid = movimientos_validos(state)
            if not valid:
                break
            if turno == 1:
                max_val = -1000
                best = valid[0]
                for r, c in valid:
                    ns = state.copy()
                    ns[r, c] = 1
                    key = estado_str(ns)
                    val = V.get(key, 1.0)
                    if val >= max_val:
                        max_val = val
                        best = (r, c)
                row, col = best
                state[row, col] = 1
                positions.append(estado_str(state))
            else:
                ix = np.random.choice(len(valid))
                row, col = valid[ix]
                state[row, col] = -1
            result = juego_terminado(state)
            if result is not None:
                break
            turno = -turno
        if result == 1:
            recompensa = 1
        elif result == 0:
            recompensa = 0.5
        else:
            recompensa = 0
        for p in reversed(positions):
            if V.get(p) is None:
                V[p] = 1.0
            V[p] += 0.5 * (recompensa - V[p])
            recompensa = V[p]
        recompensas_medias[i][exp] += (result == 1)

for ej in range(partidas):
    # UCB
    i = 2
    V = {}
    visits = {}
    t = 0
    for exp in range(turnos):
        state = np.zeros((3,3))
        positions = []
        turno = 1
        while True:
            valid = movimientos_validos(state)
            if not valid:
                break
            t += 1
            if turno == 1:
                max_val = -1000
                best = valid[0]
                for r, c in valid:
                    ns = state.copy()
                    ns[r, c] = 1
                    key = estado_str(ns)
                    q = V.get(key, 0)
                    n = visits.get(key, 1)
                    score = q + 1.0 * math.sqrt(math.log(t + 1) / n)
                    if score >= max_val:
                        max_val = score
                        best = (r, c)
                row, col = best
                state[row, col] = 1
                sk = estado_str(state)
                positions.append(sk)
                visits[sk] = visits.get(sk, 0) + 1
            else:
                ix = np.random.choice(len(valid))
                row, col = valid[ix]
                state[row, col] = -1
            result = juego_terminado(state)
            if result is not None:
                break
            turno = -turno
        if result == 1:
            recompensa = 1
        elif result == 0:
            recompensa = 0.5
        else:
            recompensa = 0
        for p in reversed(positions):
            if V.get(p) is None:
                V[p] = 0
            V[p] += 0.5 * (recompensa - V[p])
            recompensa = V[p]
        recompensas_medias[i][exp] += (result == 1)

for ej in range(partidas):
    # Gradiente
    i = 3
    H = {}
    rewards_history = []
    for exp in range(turnos):
        state = np.zeros((3,3))
        positions = []
        chosen_probs = []
        turno = 1
        while True:
            valid = movimientos_validos(state)
            if not valid:
                break
            if turno == 1:
                if len(valid) == 1:
                    row, col = valid[0]
                else:
                    next_states = []
                    for r, c in valid:
                        ns = state.copy()
                        ns[r, c] = 1
                        next_states.append(estado_str(ns))
                    prefs = [H.get(s, 0) for s in next_states]
                    pi = softmax(prefs)
                    ix = np.random.choice(len(valid), p=pi)
                    row, col = valid[ix]
                    chosen_probs.append((estado_str(state), pi[ix]))
                state[row, col] = 1
                positions.append(estado_str(state))
            else:
                ix = np.random.choice(len(valid))
                row, col = valid[ix]
                state[row, col] = -1
            result = juego_terminado(state)
            if result is not None:
                break
            turno = -turno
        if result == 1:
            recompensa = 1
        elif result == 0:
            recompensa = 0.5
        else:
            recompensa = 0
        rewards_history.append(recompensa)
        r_mean = np.mean(rewards_history)
        for p in reversed(positions):
            if H.get(p) is None:
                H[p] = 0
            pi_chosen = 0
            for s, pc in chosen_probs:
                if s == p:
                    pi_chosen = pc
                    break
            H[p] += 0.1 * (recompensa - r_mean) * (1 - pi_chosen)
            recompensa = H[p]
        recompensas_medias[i][exp] += (result == 1)

recompensas_medias /= partidas

In [ ]:
plt.figure(figsize=(10, 5))
for i, m in enumerate(metodos):
    plt.plot(recompensas_medias[i], label=m, linewidth=1.5)
plt.legend()
plt.grid(True, alpha=0.3)
plt.xlabel('Partidas')
plt.ylabel('Tasa de victorias')
plt.title('Comparación de métodos - Tres en raya')
plt.show()

## Resumen

| Método | Exploración | Selección | Actualización |
|--------|-------------|-----------|---------------|
| **ε-greedy** | Aleatoria con prob ε | $\arg\max Q(s')$ | $V(s) \leftarrow V(s) + \alpha(R - V(s))$ |
| **Optimista** | Valores iniciales altos | $\arg\max Q(s')$ con sesgo | $V(s) \leftarrow V(s) + \alpha(R - V(s))$ |
| **UCB** | Dirigida por $c\sqrt{\ln t / N(s')}$ | $Q(s') + c\sqrt{\ln t / N(s')}$ | $V(s) \leftarrow V(s) + \alpha(R - V(s))$ |
| **Gradiente** | Por softmax | $\pi(s') = \text{softmax}(H(s'))$ | $H(s) \leftarrow H(s) + \alpha(R - \bar{R})(1 - \pi)$ |